In [43]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_json, from_json, current_timestamp, get_json_object, explode, first, explode_outer
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType
import os

In [44]:
try:
    spark.stop()
except:
    pass

In [45]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

In [46]:
bronze_path = "../data_lake/bronze/batch_data/"
silver_base_path = "../data_lake/silver/"

In [47]:
bronze_df = (spark.read.parquet(bronze_path)
).withColumnRenamed("fullUrl","id")

resource_types_rows = (bronze_df.select("resource_type")
             .distinct()
             .filter(col("resource_type").isNotNull()).collect()
)
resource_types = [row.resource_type for row in resource_types_rows]

In [48]:
bronze_df.schema

StructType([StructField('bundle_resource_type', StringType(), True), StructField('bundle_type', StringType(), True), StructField('id', StringType(), True), StructField('resource', MapType(StringType(), StringType(), True), True), StructField('request', MapType(StringType(), StringType(), True), True), StructField('input_file_name', StringType(), True), StructField('ingestion_timestamp', TimestampType(), True), StructField('resource_type', StringType(), True)])

In [49]:
df = spark.read.parquet(bronze_path+"resource_type=Patient")
df.schema

StructType([StructField('bundle_resource_type', StringType(), True), StructField('bundle_type', StringType(), True), StructField('fullUrl', StringType(), True), StructField('resource', MapType(StringType(), StringType(), True), True), StructField('request', MapType(StringType(), StringType(), True), True), StructField('input_file_name', StringType(), True), StructField('ingestion_timestamp', TimestampType(), True)])

In [ ]:
def process_bronze_to_silver(resource_type):
    input_path = os.path.join(bronze_path, f"resource_type={resource_type}/")
    # print(input_path)
    output_path = os.path.join(silver_base_path, "silver_"+resource_type)
    
    df_bronze_filtered = spark.read.parquet(input_path)

    df_pivot = (df_bronze_filtered.select(
    col("fullUrl"),
    explode_outer(col("resource")).alias("key","value"))
    )

    df_pivot_2 = df_pivot.groupBy("fullUrl").pivot("key").agg(first("value")).withColumn("silver_timestamp", current_timestamp())
    # print(df_pivot_2.schema, df_bronze_filtered.schema)

    df_silver = (df_bronze_filtered.alias("a").join(df_pivot_2.alias("p"), col("a.fullUrl") == col("p.fullUrl"), how="left")
             .select(
                 col("a.bundle_resource_type"),
                    col("a.bundle_type"),
                    col("a.request"),
                    col("p.*"),
                    col("a.input_file_name")
             )
             )
    df_silver.write.mode("append").option("mergeSchema", "true").parquet(output_path)

In [51]:
for resource_type in resource_types:
    process_bronze_to_silver(resource_type)

StructType([StructField('fullUrl', StringType(), True), StructField('category', StringType(), True), StructField('code', StringType(), True), StructField('effectiveDateTime', StringType(), True), StructField('encounter', StringType(), True), StructField('id', StringType(), True), StructField('issued', StringType(), True), StructField('meta', StringType(), True), StructField('performer', StringType(), True), StructField('presentedForm', StringType(), True), StructField('resourceType', StringType(), True), StructField('result', StringType(), True), StructField('status', StringType(), True), StructField('subject', StringType(), True), StructField('silver_timestamp', TimestampType(), False)]) StructType([StructField('bundle_resource_type', StringType(), True), StructField('bundle_type', StringType(), True), StructField('fullUrl', StringType(), True), StructField('resource', MapType(StringType(), StringType(), True), True), StructField('request', MapType(StringType(), StringType(), True), T

StructType([StructField('fullUrl', StringType(), True), StructField('category', StringType(), True), StructField('code', StringType(), True), StructField('component', StringType(), True), StructField('effectiveDateTime', StringType(), True), StructField('encounter', StringType(), True), StructField('id', StringType(), True), StructField('issued', StringType(), True), StructField('meta', StringType(), True), StructField('resourceType', StringType(), True), StructField('status', StringType(), True), StructField('subject', StringType(), True), StructField('valueCodeableConcept', StringType(), True), StructField('valueQuantity', StringType(), True), StructField('valueString', StringType(), True), StructField('silver_timestamp', TimestampType(), False)]) StructType([StructField('bundle_resource_type', StringType(), True), StructField('bundle_type', StringType(), True), StructField('fullUrl', StringType(), True), StructField('resource', MapType(StringType(), StringType(), True), True), Struc

StructType([StructField('fullUrl', StringType(), True), StructField('billablePeriod', StringType(), True), StructField('careTeam', StringType(), True), StructField('claim', StringType(), True), StructField('contained', StringType(), True), StructField('created', StringType(), True), StructField('diagnosis', StringType(), True), StructField('facility', StringType(), True), StructField('id', StringType(), True), StructField('identifier', StringType(), True), StructField('insurance', StringType(), True), StructField('insurer', StringType(), True), StructField('item', StringType(), True), StructField('outcome', StringType(), True), StructField('patient', StringType(), True), StructField('payment', StringType(), True), StructField('provider', StringType(), True), StructField('referral', StringType(), True), StructField('resourceType', StringType(), True), StructField('status', StringType(), True), StructField('total', StringType(), True), StructField('type', StringType(), True), StructField

StructType([StructField('fullUrl', StringType(), True), StructField('agent', StringType(), True), StructField('id', StringType(), True), StructField('meta', StringType(), True), StructField('recorded', StringType(), True), StructField('resourceType', StringType(), True), StructField('target', StringType(), True), StructField('silver_timestamp', TimestampType(), False)]) StructType([StructField('bundle_resource_type', StringType(), True), StructField('bundle_type', StringType(), True), StructField('fullUrl', StringType(), True), StructField('resource', MapType(StringType(), StringType(), True), True), StructField('request', MapType(StringType(), StringType(), True), True), StructField('input_file_name', StringType(), True), StructField('ingestion_timestamp', TimestampType(), True)])
StructType([StructField('fullUrl', StringType(), True), StructField('billablePeriod', StringType(), True), StructField('created', StringType(), True), StructField('diagnosis', StringType(), True), StructFiel

StructType([StructField('fullUrl', StringType(), True), StructField('author', StringType(), True), StructField('category', StringType(), True), StructField('content', StringType(), True), StructField('context', StringType(), True), StructField('custodian', StringType(), True), StructField('date', StringType(), True), StructField('id', StringType(), True), StructField('identifier', StringType(), True), StructField('meta', StringType(), True), StructField('resourceType', StringType(), True), StructField('status', StringType(), True), StructField('subject', StringType(), True), StructField('type', StringType(), True), StructField('silver_timestamp', TimestampType(), False)]) StructType([StructField('bundle_resource_type', StringType(), True), StructField('bundle_type', StringType(), True), StructField('fullUrl', StringType(), True), StructField('resource', MapType(StringType(), StringType(), True), True), StructField('request', MapType(StringType(), StringType(), True), True), StructField

StructType([StructField('fullUrl', StringType(), True), StructField('encounter', StringType(), True), StructField('id', StringType(), True), StructField('identifier', StringType(), True), StructField('location', StringType(), True), StructField('numberOfInstances', StringType(), True), StructField('numberOfSeries', StringType(), True), StructField('procedureCode', StringType(), True), StructField('resourceType', StringType(), True), StructField('series', StringType(), True), StructField('started', StringType(), True), StructField('status', StringType(), True), StructField('subject', StringType(), True), StructField('silver_timestamp', TimestampType(), False)]) StructType([StructField('bundle_resource_type', StringType(), True), StructField('bundle_type', StringType(), True), StructField('fullUrl', StringType(), True), StructField('resource', MapType(StringType(), StringType(), True), True), StructField('request', MapType(StringType(), StringType(), True), True), StructField('input_file

StructType([StructField('fullUrl', StringType(), True), StructField('authoredOn', StringType(), True), StructField('category', StringType(), True), StructField('dosageInstruction', StringType(), True), StructField('encounter', StringType(), True), StructField('id', StringType(), True), StructField('intent', StringType(), True), StructField('medicationCodeableConcept', StringType(), True), StructField('medicationReference', StringType(), True), StructField('meta', StringType(), True), StructField('reasonCode', StringType(), True), StructField('reasonReference', StringType(), True), StructField('requester', StringType(), True), StructField('resourceType', StringType(), True), StructField('status', StringType(), True), StructField('subject', StringType(), True), StructField('silver_timestamp', TimestampType(), False)]) StructType([StructField('bundle_resource_type', StringType(), True), StructField('bundle_type', StringType(), True), StructField('fullUrl', StringType(), True), StructField

StructType([StructField('fullUrl', StringType(), True), StructField('context', StringType(), True), StructField('dosage', StringType(), True), StructField('effectiveDateTime', StringType(), True), StructField('id', StringType(), True), StructField('medicationCodeableConcept', StringType(), True), StructField('reasonCode', StringType(), True), StructField('reasonReference', StringType(), True), StructField('resourceType', StringType(), True), StructField('status', StringType(), True), StructField('subject', StringType(), True), StructField('silver_timestamp', TimestampType(), False)]) StructType([StructField('bundle_resource_type', StringType(), True), StructField('bundle_type', StringType(), True), StructField('fullUrl', StringType(), True), StructField('resource', MapType(StringType(), StringType(), True), True), StructField('request', MapType(StringType(), StringType(), True), True), StructField('input_file_name', StringType(), True), StructField('ingestion_timestamp', TimestampType(